## Elevation and Gradient from Latitude/Longitude

This notebook builds a dataset of **elevation** and **gradient (slope)** for points given their latitude and longitude.

### Two main approaches

1. **API-based (here)** – Use a free elevation API (e.g. [Open-Elevation](https://open-elevation.com/)) to get elevation at each point, then estimate gradient by querying elevation at four neighboring points and computing slope (rise over run). Good for moderate numbers of points; rate limits may apply.

2. **DEM-based** – Use a global Digital Elevation Model (e.g. SRTM 30 m, Copernicus DEM) with `rasterio` (and optionally the `elevation` package to download tiles). Sample elevation at your points, then compute slope from the DEM grid (e.g. with `numpy.gradient` or `richdem`) and sample slope at the same points. Better for large or repeated runs and no external API.

Below we implement the **API-based** approach and add an optional **DEM-based** path using a single SRTM tile for your region.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

# Optional: for DEM-based approach
try:
    import rasterio
    from rasterio.sample import sample_gen
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False

### 1. Load points (Latitude, Longitude)

Use any CSV that has `Latitude` and `Longitude` columns. For many points, consider taking **unique (lat, lon)** first to avoid duplicate API calls.

In [2]:
# Example: load submission template or your training/validation CSV
df = pd.read_csv("landsat_features_training.csv")

# Work with unique locations to minimize API calls
locations = df[["Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True)
print(f"Unique locations: {len(locations)}")

Unique locations: 162


### 2. API-based: elevation via Open-Elevation

Batch requests (e.g. 100–500 points per request) to stay within limits. Open-Elevation free tier has limits; for large datasets consider self-hosting or the DEM approach.

In [3]:
OPEN_ELEVATION_URL = "https://api.open-elevation.com/api/v1/lookup"
BATCH_SIZE = 100  # reduce if you hit rate limits


def fetch_elevation_batch(lats, lons):
    """Request elevation for a list of (lat, lon). Returns list of elevations (m)."""
    payload = {
        "locations": [
            {"latitude": float(lat), "longitude": float(lon)}
            for lat, lon in zip(lats, lons)
        ]
    }
    try:
        r = requests.post(OPEN_ELEVATION_URL, json=payload, timeout=30)
        r.raise_for_status()
        data = r.json()
        return [x["elevation"] for x in data.get("results", [])]
    except Exception as e:
        print(f"Request failed: {e}")
        return [np.nan] * len(lats)


def add_elevation_api(locations_df):
    """Add 'elevation_m' column using Open-Elevation API."""
    lats = locations_df["Latitude"].tolist()
    lons = locations_df["Longitude"].tolist()
    elevations = []
    for i in tqdm(range(0, len(lats), BATCH_SIZE), desc="Elevation"):
        batch_lats = lats[i : i + BATCH_SIZE]
        batch_lons = lons[i : i + BATCH_SIZE]
        elev = fetch_elevation_batch(batch_lats, batch_lons)
        elevations.extend(elev)
    out = locations_df.copy()
    out["elevation_m"] = elevations
    return out

### 3. Gradient (slope) from neighboring elevations

For each point we query elevation at the center and at four offsets (N, S, E, W). We convert latitude/longitude deltas to approximate meters, then compute slope as **rise over run** (gradient magnitude in m/m), and optionally express it as **slope_degrees** (arctan of gradient) or **slope_percent** (100 × gradient).

In [4]:
# Approximate meters per degree at a given latitude (WGS84)
def m_per_deg_lat(lat):
    return 111320.0  # roughly constant


def m_per_deg_lon(lat):
    return 111320.0 * np.cos(np.radians(lat))


def gradient_from_neighbors(lat, lon, d_deg=0.0005):
    """
    Get elevation at (lat, lon) and at 4 neighbors; return elevation (m) and slope (m/m).
    d_deg: offset in degrees (~50 m at mid-latitudes if 0.0005).
    """
    points = [
        (lat, lon),           # center
        (lat + d_deg, lon),   # N
        (lat - d_deg, lon),   # S
        (lat, lon + d_deg),   # E
        (lat, lon - d_deg),   # W
    ]
    lats = [p[0] for p in points]
    lons = [p[1] for p in points]
    elevs = fetch_elevation_batch(lats, lons)
    if len(elevs) != 5 or any(np.isnan(elevs)):
        return np.nan, np.nan
    z_c, z_n, z_s, z_e, z_w = elevs
    dx_m = m_per_deg_lon(lat) * (2 * d_deg)
    dy_m = m_per_deg_lat(lat) * (2 * d_deg)
    dz_dx = (z_e - z_w) / dx_m if dx_m > 0 else 0.0
    dz_dy = (z_n - z_s) / dy_m if dy_m > 0 else 0.0
    slope_m_per_m = np.sqrt(dz_dx**2 + dz_dy**2)
    return z_c, slope_m_per_m


def add_gradient_api(locations_df, d_deg=0.0005):
    """Add elevation_m and slope (gradient magnitude, m/m) using neighbor-based method."""
    elevations = []
    slopes = []
    for _, row in tqdm(locations_df.iterrows(), total=len(locations_df), desc="Gradient"):
        z, slope = gradient_from_neighbors(row["Latitude"], row["Longitude"], d_deg=d_deg)
        elevations.append(z)
        slopes.append(slope)
    out = locations_df.copy()
    out["elevation_m"] = elevations
    out["slope_m_per_m"] = slopes
    out["slope_degrees"] = np.degrees(np.arctan(slopes))
    out["slope_percent"] = 100.0 * np.array(slopes)
    return out

In [9]:
locations_test = locations[:5]

locations_test = add_elevation_api(locations_df=locations_test)
locations_test.head()

Elevation: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]


,Latitude,Longitude,elevation_m
0,-28.760833,17.730278,166.0
1,-26.861111,28.884722,1526.0
2,-26.450000,28.085833,1475.0
3,-27.671111,27.236944,1345.0
4,-27.356667,27.286389,1360.0


In [10]:
locations_test = add_gradient_api(locations_df=locations_test)
locations_test.head()

Gradient: 100%|██████████| 5/5 [14:48<00:00, 177.79s/it]


,Latitude,Longitude,elevation_m,slope_m_per_m,slope_degrees,slope_percent
0,-28.760833,17.730278,166.0,0.089831,5.133166,8.983112
1,-26.861111,28.884722,1526.0,0.000000,0.000000,0.000000
2,-26.450000,28.085833,1475.0,0.010033,0.574851,1.003337
3,-27.671111,27.236944,1345.0,0.080848,4.622196,8.084801
4,-27.356667,27.286389,1360.0,0.026949,1.543710,2.694934


### 4. Run and merge back

Choose one:
- **Elevation only** (fewer API calls).
- **Elevation + gradient** (5× more calls per unique location).

In [5]:
# Option A: elevation only (1 request per unique location)
# locs_with_elev = add_elevation_api(locations)

# Option B: elevation + gradient (5 requests per unique location)
locs_with_elev_slope = add_gradient_api(locations, d_deg=0.0005)

# Merge back to full dataframe (if you have multiple rows per location)
df_merged = df.merge(
    locs_with_elev_slope[["Latitude", "Longitude", "elevation_m", "slope_m_per_m", "slope_degrees", "slope_percent"]],
    on=["Latitude", "Longitude"],
    how="left"
)
df_merged.head(10)

Gradient: 100%|██████████| 162/162 [02:20<00:00,  1.16it/s]


,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI,elevation_m,slope_m_per_m,slope_degrees,slope_percent
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,166.0,0.089831,5.133166,8.983112
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,1526.0,0.000000,0.000000,0.000000
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,1475.0,0.010033,0.574851,1.003337
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,1345.0,0.080848,4.622196,8.084801
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,1360.0,0.026949,1.543710,2.694934
5,-27.010111,26.698083,04-01-2011,12433.5,10433.5,9579.5,8531.5,0.129651,0.042672,1291.0,0.010083,0.577687,1.008289
6,-25.127778,27.628889,04-01-2011,7814.0,5189.5,6664.0,5222.0,0.079431,-0.124394,957.0,0.019844,1.136841,1.984422
7,-25.206390,27.558000,04-01-2011,14137.5,10466.5,10315.5,8536.0,0.156300,0.007266,957.0,0.039714,2.274251,3.971403
8,-24.695140,27.409060,04-01-2011,15543.0,10647.0,11919.5,9642.5,0.131944,-0.056389,912.0,0.000000,0.000000,0.000000
9,-26.984722,26.632278,04-01-2011,13683.0,10207.0,14011.5,11850.5,-0.011862,-0.157091,1289.0,0.000000,0.000000,0.000000


In [6]:
# Save dataset with elevation and gradient
out_path = "elevation_gradient_locations.csv"
locs_with_elev_slope.to_csv(out_path, index=False)
print(f"Saved {len(locs_with_elev_slope)} rows to {out_path}")

Saved 162 rows to elevation_gradient_locations.csv


### 5. Optional: DEM-based approach (rasterio)

If you have a DEM GeoTIFF for your region (e.g. from [USGS Earth Explorer](https://earthexplorer.usgs.gov/) SRTM, or from the `elevation` package: `eio clip -o dem.tif --bounds left bottom right top`), you can sample elevation at points and compute slope from the raster. Example below assumes you have `dem.tif` in the same folder.

In [ ]:
if HAS_RASTERIO:
    DEM_PATH = "dem.tif"  # create e.g. with: eio clip -o dem.tif --bounds lon_min lat_min lon_max lat_max

    def sample_elevation_dem(locations_df, dem_path):
        """Sample elevation at (lon, lat) from a DEM. Returns elevations in meters."""
        with rasterio.open(dem_path) as src:
            coords = [(row["Longitude"], row["Latitude"]) for _, row in locations_df.iterrows()]
            elevs = [v[0] for v in sample_gen(src, coords)]
        out = locations_df.copy()
        out["elevation_m"] = elevs
        return out

    def slope_from_dem(dem_path):
        """Compute slope raster (rise/run) from DEM using numpy gradient; return (elev_array, slope_array, transform)."""
        with rasterio.open(dem_path) as src:
            elev = src.read(1).astype(np.float64)
            nodata = src.nodata
            if nodata is not None:
                elev[elev == nodata] = np.nan
            transform = src.transform
        # Pixel size in m (approximate for geographic DEM)
        res_x = abs(transform.a)
        res_y = abs(transform.e)
        dy_dx, dy_dy = np.gradient(elev, res_x, -res_y)
        slope = np.sqrt(dy_dx**2 + dy_dy**2)
        return elev, slope, transform

    # Uncomment and run when dem.tif exists:
    # elev_array, slope_array, transform = slope_from_dem(DEM_PATH)
    # Then use rasterio.sample.sample_gen with slope_array + transform to get slope at each (lon,lat)
else:
    print("rasterio not available; install it for DEM-based sampling.")

### Summary

| Approach | Elevation | Gradient | Best for |
|----------|-----------|----------|----------|
| **API (Open-Elevation)** | Yes, batch | Yes, via 4 neighbors | Small/medium unique point sets; quick to try |
| **DEM + rasterio** | Sample at points | Compute slope raster, then sample | Large or repeated runs; full control; no API limits |

For gradient we use a small offset `d_deg` (~0.0005° ≈ 50 m) so slope is representative of the immediate terrain. Increase `d_deg` for a smoother, more regional slope.